# Bias Mitigation in Chest X-Ray dataset

In this notebook we are going to explore, and apply methods of bias mitigation to one part of Chest X-Ray dataset from Kaggle.

This notebook contains the full pipeline to 
 - view/analyse the data
 - clean up artefacts and extract sensitive attributes
 - analyse bias metrics
 - mitigate bias with preprocessing methods

In [1]:
## Imports
import os
import numpy as np
import pandas as pd
import fairlearn
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

### Loading the dataset

`The dataset CSV file is assumed to be be inside data folder which is in the same directory as the notebook`

In [2]:
## Loading the dataset from the CSV file
file_path = os.path.join("data", "Bareyan_Sipan.csv")

data = pd.read_csv(file_path)

In [3]:
print("Data Columns and Types:")
print(data.dtypes)

Data Columns and Types:
Image Index                     object
Finding Labels                  object
Follow-up #                      int64
Patient ID                       int64
Patient Age                      int64
Patient Gender                  object
View Position                   object
OriginalImage[Width              int64
Height]                          int64
OriginalImagePixelSpacing[x    float64
y]                             float64
dtype: object


In [4]:
data.head(10)

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171
4,00000005_000.png,No Finding,0,5,69,F,PA,2048,2500,0.168,0.168
5,00000005_001.png,No Finding,1,5,69,F,AP,2500,2048,0.168,0.168
6,00000005_002.png,No Finding,2,5,69,F,AP,2500,2048,0.168,0.168
7,00000005_003.png,No Finding,3,5,69,F,PA,2992,2991,0.143,0.143
8,00000005_004.png,No Finding,4,5,70,F,PA,2986,2991,0.143,0.143
9,00000005_005.png,No Finding,5,5,70,F,PA,2514,2991,0.143,0.143


## Data Cleanup and Initial Preprocessing

In this section we fix anomalies in the data, normalize the dataset and do initial data visualization

### Fixing Column names

In [5]:
## Cleaning column names
cleaned_data = data.copy()

rename_map = [
    ("OriginalImage[Width", "Width"),
    ("Height]", "Height"),
    ("OriginalImagePixelSpacing[x", "SpacingX"),
    ("y]", "SpacingY"),
]
for k, v in rename_map:
    if k in cleaned_data.columns:
        cleaned_data = cleaned_data.rename(columns={k: v})

print("Labels after cleaning:")
print(cleaned_data.dtypes)

Labels after cleaning:
Image Index        object
Finding Labels     object
Follow-up #         int64
Patient ID          int64
Patient Age         int64
Patient Gender     object
View Position      object
Width               int64
Height              int64
SpacingX          float64
SpacingY          float64
dtype: object


### `Patient Age` Column

#### Anomalies

In [6]:
## Calculating statistics for the "Patient Age" column

# Calculate minimum and maximum age
mn_age = cleaned_data["Patient Age"].min()
mx_age = cleaned_data["Patient Age"].max()

# Find ages less than 0 and more than 100 to identify potential data entry errors
less_than_0 = (cleaned_data["Patient Age"] < 0).sum()
more_than_100 = (cleaned_data["Patient Age"] > 100).sum()

# Find NA values
na_values = cleaned_data.isna().sum().sum()


print(f"Minimum age: {mn_age}, Maximum age: {mx_age}")
print("Ages less than 0:", less_than_0)
print("Ages more than 100:", more_than_100)
print("NA values", na_values)

Minimum age: 1, Maximum age: 414
Ages less than 0: 0
Ages more than 100: 6
NA values 0


In [7]:
## Deleting anomalies in the "Patient Age" column, by removing rows with ages > 100
cleaned_data = cleaned_data[cleaned_data["Patient Age"] <= 100]

#### Creating age categories

In [8]:
# Divide patient ages into specified number of categories

TARGET_CATEGORIES = 6

# Divide the "Patient Age" column into TARGET_CATEGORIES categories, and create a new column "Age Category" with the category labels mentioning the range of ages in each category
cleaned_data["Age Category"] = pd.cut(cleaned_data["Patient Age"], bins=TARGET_CATEGORIES, labels=[f"Category {i+1}" for i in range(TARGET_CATEGORIES)])

#find min and max age in each category, and rename the category labels to include the age range
age_ranges = cleaned_data.groupby("Age Category", observed=True)["Patient Age"].agg(["min", "max"])
cleaned_data["Age Category"] = cleaned_data["Age Category"].cat.rename_categories([f"{row['min']}-{row['max']}" for _, row in age_ranges.iterrows()])

for category in sorted(set(cleaned_data["Age Category"]), key=lambda x: int(x.split("-")[0])):
    print(f"Category: {category}, Count: {(cleaned_data['Age Category'] == category).sum()}")

Category: 1-16, Count: 2158
Category: 17-31, Count: 8696
Category: 32-47, Count: 14051
Category: 48-62, Count: 18806
Category: 63-77, Count: 9031
Category: 78-93, Count: 824


### Data Normalization

In [9]:
# Cleaning up and normalizing columns in the dataset

# Normalize numerical Columns
num_cols = ["Patient Age", "Follow-up #", "Width", "Height", "SpacingX", "SpacingY"]
for col in num_cols:
    if col in cleaned_data.columns:
        cleaned_data[col] = pd.to_numeric(cleaned_data[col], errors="coerce")

# Drop NAs
cleaned_data = cleaned_data.dropna(subset=["Patient Age", "Patient Gender", "View Position", "Finding Labels", "Patient ID"])


# Normalize Categorical Data
cleaned_data["Patient Gender"] = cleaned_data["Patient Gender"].astype(str).str.strip().str.upper()
cleaned_data["View Position"]  = cleaned_data["View Position"].astype(str).str.strip().str.upper()

#### Create binary protected attributes

In [10]:
# Binary protected attrs: Gender(Male privileged), Age(60+ privileged), View Position(AP privileged)
cleaned_data["Gender_male"] = (cleaned_data["Patient Gender"] == "M").astype(int)
cleaned_data["Age_60plus"] = (cleaned_data["Patient Age"] >= 60).astype(int)
cleaned_data["View_AP"] = (cleaned_data["View Position"] == "AP").astype(int)


cleaned_data.head(10)

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,Width,Height,SpacingX,SpacingY,Age Category,Gender_male,Age_60plus,View_AP
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,48-62,1,0,0
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,48-62,1,0,0
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,48-62,1,0,0
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,78-93,1,1,0
4,00000005_000.png,No Finding,0,5,69,F,PA,2048,2500,0.168,0.168,63-77,0,1,0
5,00000005_001.png,No Finding,1,5,69,F,AP,2500,2048,0.168,0.168,63-77,0,1,1
6,00000005_002.png,No Finding,2,5,69,F,AP,2500,2048,0.168,0.168,63-77,0,1,1
7,00000005_003.png,No Finding,3,5,69,F,PA,2992,2991,0.143,0.143,63-77,0,1,0
8,00000005_004.png,No Finding,4,5,70,F,PA,2986,2991,0.143,0.143,63-77,0,1,0
9,00000005_005.png,No Finding,5,5,70,F,PA,2514,2991,0.143,0.143,63-77,0,1,0


### `Findings` Column

From this part we can proceed in two different paths. 
- We can create a new column in the dataset for each finding, this way we can do bias analysis for each one of the possible findings
- We keep a binary column: Finding or no Finding

We will split the dataset both ways, will continue analysis in hybrid

#### Path 1: Column for each finding

In [11]:
## Creating one-hot encoded labeled data
labeled_data = cleaned_data.copy()

# Finding unique labels
all_labels = set()
for labels in labeled_data["Finding Labels"]:
    for label in labels.split("|"):
        all_labels.add(label)

print(f"Found {len(all_labels)} possible labels:")
for label in all_labels:
    print(label)

all_labels.remove("No Finding")
# Adding column for each label
for label in all_labels:
    labeled_data[label] = labeled_data["Finding Labels"].apply(lambda x: 1 if label in x.split("|") else 0)

labeled_data.drop(columns=["Finding Labels"], inplace=True)

labeled_data.head(10)

Found 15 possible labels:
Emphysema
Effusion
Fibrosis
Pneumonia
Pneumothorax
Atelectasis
Nodule
Mass
Infiltration
Edema
Consolidation
No Finding
Cardiomegaly
Hernia
Pleural_Thickening


,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,Width,Height,SpacingX,SpacingY,...,Pneumothorax,Atelectasis,Nodule,Mass,Infiltration,Edema,Consolidation,Cardiomegaly,Hernia,Pleural_Thickening
0,00000001_000.png,0,1,58,M,PA,2682,2749,0.143,0.143,...,0,0,0,0,0,0,0,1,0,0
1,00000001_001.png,1,1,58,M,PA,2894,2729,0.143,0.143,...,0,0,0,0,0,0,0,1,0,0
2,00000001_002.png,2,1,58,M,PA,2500,2048,0.168,0.168,...,0,0,0,0,0,0,0,1,0,0
3,00000002_000.png,0,2,81,M,PA,2500,2048,0.171,0.171,...,0,0,0,0,0,0,0,0,0,0
4,00000005_000.png,0,5,69,F,PA,2048,2500,0.168,0.168,...,0,0,0,0,0,0,0,0,0,0
5,00000005_001.png,1,5,69,F,AP,2500,2048,0.168,0.168,...,0,0,0,0,0,0,0,0,0,0
6,00000005_002.png,2,5,69,F,AP,2500,2048,0.168,0.168,...,0,0,0,0,0,0,0,0,0,0
7,00000005_003.png,3,5,69,F,PA,2992,2991,0.143,0.143,...,0,0,0,0,0,0,0,0,0,0
8,00000005_004.png,4,5,70,F,PA,2986,2991,0.143,0.143,...,0,0,0,0,0,0,0,0,0,0
9,00000005_005.png,5,5,70,F,PA,2514,2991,0.143,0.143,...,0,0,0,0,0,0,0,0,0,0


In [12]:
## Display value counts for each label in a compact table
label_counts = {}
for label in all_labels:
    label_counts[label] = labeled_data[label].value_counts().to_dict()

# Create a DataFrame for better visualization
counts_df = pd.DataFrame(label_counts).T
counts_df.columns = ["No Finding", "Finding"]
print(counts_df)

                    No Finding  Finding
Emphysema                52336     1230
Effusion                 47241     6325
Fibrosis                 52711      855
Pneumonia                52899      667
Pneumothorax             50828     2738
Atelectasis              47996     5570
Nodule                   50461     3105
Mass                     50666     2900
Infiltration             44405     9161
Edema                    52582      984
Consolidation            51466     2100
Cardiomegaly             52300     1266
Hernia                   53450      116
Pleural_Thickening       51881     1685


#### Path 2: Binary Label

In [13]:
## Creating binary labeled data
binary_labeled_data = cleaned_data.copy()
binary_labeled_data["Finding"] = binary_labeled_data["Finding Labels"].apply(lambda x: 0 if x=="No Finding" else 1)

binary_labeled_data.drop(columns=["Finding Labels"], inplace=True)

binary_labeled_data.head(10)

,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,Width,Height,SpacingX,SpacingY,Age Category,Gender_male,Age_60plus,View_AP,Finding
0,00000001_000.png,0,1,58,M,PA,2682,2749,0.143,0.143,48-62,1,0,0,1
1,00000001_001.png,1,1,58,M,PA,2894,2729,0.143,0.143,48-62,1,0,0,1
2,00000001_002.png,2,1,58,M,PA,2500,2048,0.168,0.168,48-62,1,0,0,1
3,00000002_000.png,0,2,81,M,PA,2500,2048,0.171,0.171,78-93,1,1,0,0
4,00000005_000.png,0,5,69,F,PA,2048,2500,0.168,0.168,63-77,0,1,0,0
5,00000005_001.png,1,5,69,F,AP,2500,2048,0.168,0.168,63-77,0,1,1,0
6,00000005_002.png,2,5,69,F,AP,2500,2048,0.168,0.168,63-77,0,1,1,0
7,00000005_003.png,3,5,69,F,PA,2992,2991,0.143,0.143,63-77,0,1,0,0
8,00000005_004.png,4,5,70,F,PA,2986,2991,0.143,0.143,63-77,0,1,0,0
9,00000005_005.png,5,5,70,F,PA,2514,2991,0.143,0.143,63-77,0,1,0,0


In [14]:
## Display value counts for binary findings
print("Value counts for Binary Finding:")
print(binary_labeled_data["Finding"].value_counts())

Value counts for Binary Finding:
Finding
0    28923
1    24643
Name: count, dtype: int64


### Data Visualization 

In this section we will explore the data more, using plots

In this first part we treat the records individually, not accounting that a single patient can have multiple records associated with him

#### Overall Statistics

The following Pie Chart shows the percentages of records that resulted in a finding vs no finding at all

In [15]:
counts = binary_labeled_data["Finding"].value_counts()
df_pie = pd.DataFrame({"Status": ["No Finding", "Finding"], "Count": [counts.get(0, 0), counts.get(1, 0)]})
fig = px.pie(df_pie, values="Count", names="Status", title="Distribution of Findings", color_discrete_map={"No Finding": "green", "Finding": "red"})
fig.show()

The following Pie Charts shows the percentage of records with a finding vs no finding 
**for each finding type**

In [16]:
rows = 2
cols = (len(all_labels) + rows - 1) // rows

fig = make_subplots(
    rows=rows, cols=cols,
    specs=[[{"type": "pie"}] * cols for _ in range(rows)],
    subplot_titles=list(all_labels)
)

subplot_row = 1
subplot_col = 1

for label in all_labels:
    counts = labeled_data[label].value_counts()
    fig.add_trace(
        go.Pie(
            labels=["No Finding", "Finding"],
            values=[counts.get(0, 0), counts.get(1, 0)],
            marker=dict(colors=["green", "red"]),
            name=label
        ),
        row=subplot_row, col=subplot_col
    )
    
    subplot_col += 1
    if subplot_col > cols:
        subplot_col = 1
        subplot_row += 1

fig.update_layout(height=300*rows,width=1200, showlegend=False)
fig.show()

#### Categorical statistics

In [52]:
### Defining a function to pretty plot with pie charts the distributions of target variables 

def plot_categorical_target_distribution(dataset, target, column):
    targets = [target] if isinstance(target, str) else list(target)
    if not targets:
        raise ValueError("target must be a string or a non-empty list")

    cat_values = sorted(dataset[column].dropna().unique())
    if not cat_values:
        raise ValueError(f"No non-null values in '{column}'")

    cols = min(3, len(cat_values))
    rows = (len(cat_values) + cols - 1) // cols

    fig = make_subplots(
        rows=rows,
        cols=cols,
        specs=[[{"type": "pie"}] * cols for _ in range(rows)],
        subplot_titles=[f"{column}: {val}\n." for val in cat_values],
    )

    # Track exact trace indices per target
    trace_ids_by_target = {t: [] for t in targets}

    for t_i, t in enumerate(targets):
        r, c = 1, 1
        for v_i, val in enumerate(cat_values):
            data_subset = dataset[dataset[column] == val]
            counts = data_subset[t].value_counts()

            fig.add_trace(
                go.Pie(
                    labels=["No Finding", "Finding"],
                    values=[counts.get(0, 0), counts.get(1, 0)],
                    marker=dict(colors=["green", "red"]),
                    showlegend=(v_i == 0),
                    rotation=90,
                    visible=(t_i == 0),
                ),
                row=r, col=c
            )

            trace_ids_by_target[t].append(len(fig.data) - 1)

            c += 1
            if c > cols:
                c = 1
                r += 1

    #  Buttons
    if len(targets) > 1:
        n_traces = len(fig.data)
        buttons = []

        for t in targets:
            vis = [False] * n_traces
            for idx in trace_ids_by_target[t]:
                vis[idx] = True

            buttons.append(
                dict(
                    label=str(t),
                    method="update",
                    args=[
                        {"visible": vis},
                        {"title": {"text": f"Target: {t}"}},
                    ],
                )
            )

        # Put buttons 
        fig.update_layout(
            updatemenus=[dict(
                type="buttons",
                direction="down",
                buttons=buttons,
                showactive=True,
                x=-0.3,
                y=1.0,
                xanchor="left",
                yanchor="top",
                pad={"r": 10, "t": 10, "b": 10},
            )],
            margin=dict(r=220)
        )

    fig.update_layout(height=max(300 * rows, len(targets) * 50), title_text=f"Target: {targets[0]}")
    fig.show()


**Binary Data**

In [53]:
plot_categorical_target_distribution(binary_labeled_data, ["Finding"], "Patient Gender")

In [54]:
plot_categorical_target_distribution(binary_labeled_data, ["Finding"], "Age Category")

**Multi-Label Data**

In [55]:
plot_categorical_target_distribution(labeled_data, all_labels, "Patient Gender")

In [56]:
plot_categorical_target_distribution(labeled_data, all_labels, "Age Category")

### Patient Level Data Visualization

In [22]:
### Explore patient structure
print(f"Total images (records): {len(binary_labeled_data)}")
print(f"Unique patients: {binary_labeled_data['Patient ID'].nunique()}")
print(f"Average images per patient: {len(binary_labeled_data) / binary_labeled_data['Patient ID'].nunique():.2f}")

# Distribution of follow-ups per patient
followup_counts = binary_labeled_data.groupby("Patient ID").size()
print(f"\nFollow-ups per patient distribution:")
print(followup_counts.describe())

# Histogram of images per patient
fig = px.histogram(followup_counts, nbins=100, title="Distribution of Images per Patient",
                   labels={"value": "Number of Images", "count": "Number of Patients"})
fig.update_layout(showlegend=False)
fig.show()

Total images (records): 53566
Unique patients: 15000
Average images per patient: 3.57

Follow-ups per patient distribution:
count    15000.000000
mean         3.571067
std          6.851964
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        158.000000
dtype: float64


**For simplicity and ease of analysis we treat the Binary Labeled Data**

We create a new dataset, where we aggregate the data based on patient ids

In [23]:
### Create aggregated dataset
binary_patient_dataset = binary_labeled_data.groupby("Patient ID").agg({
    "Finding": "max",  # 1 if ANY image has finding
    "Patient Age": "first",  # Same for all images of a patient
    "Patient Gender": "first",
    "Age Category": "first",
    "Gender_male": "first",
    "Age_60plus": "first",
    "View_AP": "max",  # 1 if ANY image is AP view
    "View Position": lambda x: "Both" if x.nunique() > 1 else x.iloc[0],
    "Image Index": "count"  # Number of images
}).rename(columns={"Image Index": "Num_Images", "View_AP": "has_AP"})

binary_patient_dataset = binary_patient_dataset.reset_index()

print(f"Patient-level dataset: {len(binary_patient_dataset)} patients")
print(f"\nFinding distribution (patient-level):")
print(binary_patient_dataset["Finding"].value_counts())
print(f"\nView Position distribution (patient-level):")
print(binary_patient_dataset["View Position"].value_counts())
binary_patient_dataset.head(10)

Patient-level dataset: 15000 patients

Finding distribution (patient-level):
Finding
0    8052
1    6948
Name: count, dtype: int64

View Position distribution (patient-level):
View Position
PA      10623
Both     3438
AP        939
Name: count, dtype: int64


,Patient ID,Finding,Patient Age,Patient Gender,Age Category,Gender_male,Age_60plus,has_AP,View Position,Num_Images
0,1,1,58,M,48-62,1,0,0,PA,3
1,2,0,81,M,78-93,1,1,0,PA,1
2,5,1,69,F,63-77,0,1,1,Both,8
3,8,1,69,F,63-77,0,1,0,PA,3
4,9,1,73,M,63-77,1,1,0,PA,1
5,13,1,61,M,48-62,1,1,1,Both,47
6,16,0,71,M,63-77,1,1,0,PA,1
7,18,0,75,M,63-77,1,1,0,PA,1
8,21,1,78,M,78-93,1,1,0,PA,2
9,22,1,48,M,48-62,1,0,0,PA,2


We redo all of the same plots

#### Overall Statistics

In [24]:
counts = binary_patient_dataset["Finding"].value_counts()
df_pie = pd.DataFrame({"Status": ["No Finding", "Finding"], "Count": [counts.get(0, 0), counts.get(1, 0)]})
fig = px.pie(df_pie, values="Count", names="Status", title="Distribution of Findings", color_discrete_map={"No Finding": "green", "Finding": "red"})
fig.show()

In [25]:
rows = 2
cols = (len(all_labels) + rows - 1) // rows

fig = make_subplots(
    rows=rows, cols=cols,
    specs=[[{"type": "pie"}] * cols for _ in range(rows)],
    subplot_titles=list(all_labels)
)

subplot_row = 1
subplot_col = 1

for label in all_labels:
    counts = labeled_data[label].value_counts()
    fig.add_trace(
        go.Pie(
            labels=["No Finding", "Finding"],
            values=[counts.get(0, 0), counts.get(1, 0)],
            marker=dict(colors=["green", "red"]),
            name=label
        ),
        row=subplot_row, col=subplot_col
    )
    
    subplot_col += 1
    if subplot_col > cols:
        subplot_col = 1
        subplot_row += 1

fig.update_layout(height=300*rows,width=1200, showlegend=False)
fig.show()

#### Categorical statistics

In [26]:
plot_categorical_target_distribution(binary_patient_dataset, ["Finding"], "Patient Gender")

In [27]:
plot_categorical_target_distribution(binary_patient_dataset, ["Finding"], "Age Category")

We can remark that with the new patient level dataset the bias is more visible. We clearly see the bias in the the dataset for the age category. 

## Univariate Analysis

In this section we are analyzing numerical data and the relationship between numerical columns and the target
(We are continuing the analysis with the binary labeled data: Finding vs no Finding)

### Correlation

**Pearson correlation**

In [28]:
### Correlation analysis
num_features = ["Patient Age", "Width", "Height", "SpacingX", "SpacingY", "Follow-up #"]
correlations_img = {}
for feature in num_features:
    if feature in binary_labeled_data.columns:
        corr = np.corrcoef(binary_labeled_data[feature], 
                          binary_labeled_data.loc[binary_labeled_data[feature].notna(), "Finding"])[0, 1]
        correlations_img[feature] = corr
        print(f"{feature}: {corr:.4f}")

# Correlation matrix heatmap for image-level data
corr_cols = ["Patient Age", "Finding", "Gender_male", "View_AP"]
corr_matrix = binary_labeled_data[corr_cols].corr()

fig = px.imshow(corr_matrix, text_auto=".3f", 
                title="Correlation Matrix (Binary Labeled Data)",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
               )
fig.show()

Patient Age: 0.0789
Width: 0.0668
Height: -0.0012
SpacingX: -0.0395
SpacingY: -0.0395
Follow-up #: 0.1861


In [29]:
### Correlation analysis
num_features = ["Patient Age", "Width", "Height", "SpacingX", "SpacingY", "Num_Images"]
correlations_img = {}
for feature in num_features:
    if feature in binary_patient_dataset.columns:
        corr = np.corrcoef(binary_patient_dataset[feature], 
                          binary_patient_dataset.loc[binary_patient_dataset[feature].notna(), "Finding"])[0, 1]
        correlations_img[feature] = corr
        print(f"{feature}: {corr:.4f}")

# Correlation matrix heatmap for image-level data
corr_cols = ["Patient Age", "Finding", "Gender_male", "has_AP", "Num_Images"]
corr_matrix = binary_patient_dataset[corr_cols].corr()

fig = px.imshow(corr_matrix, text_auto=".3f", 
                title="Correlation Matrix (Binary Patient Dataset)",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
               )
fig.show()

Patient Age: 0.1724
Num_Images: 0.3330


### Age Distributions

In [30]:
### Age distribution by Finding - Binary Labeled Data
binary_labeled_data["Finding_Label"] = binary_labeled_data["Finding"].map({0: "No Finding", 1: "Finding"})

fig = px.histogram(binary_labeled_data, x="Patient Age", color="Finding_Label", 
                   barmode="stack", nbins=50,
                   title="Age Distribution by Finding (Binary Labeled Data)",
                   color_discrete_map={"No Finding": "green", "Finding": "red"},
                   category_orders={"Finding_Label": ["No Finding", "Finding"]})
fig.update_layout(legend_title_text="Finding Status")
fig.show()

### Age distribution by Finding - Patient Level
binary_patient_dataset["Finding_Label"] = binary_patient_dataset["Finding"].map({0: "No Finding", 1: "Finding"})

fig = px.histogram(binary_patient_dataset, x="Patient Age", color="Finding_Label", 
                   barmode="stack", nbins=50,
                   title="Age Distribution by Finding (Binary Patient Dataset)",
                   color_discrete_map={"No Finding": "green", "Finding": "red"},
                   category_orders={"Finding_Label": ["No Finding", "Finding"]})
fig.update_layout(legend_title_text="Finding Status")
fig.show()


We can see that in the ~first half, findings are less then no findings, but then it mirrors. This also indicates about the presence of bias

## Bivariat Analysis

### Protected attribute Joint Distributions

In [31]:
### Contingency tables - Original Dataset

# Gender × Finding
print("Gender x Finding")
ct_gender_finding_img = pd.crosstab(binary_labeled_data["Patient Gender"], binary_labeled_data["Finding"], margins=True)
ct_gender_finding_img.columns = ["No Finding", "Finding", "Total"]
print(ct_gender_finding_img)

# Age Category × Finding
print("\nAge Category x Finding")
ct_age_finding_img = pd.crosstab(binary_labeled_data["Age Category"], binary_labeled_data["Finding"], margins=True)
ct_age_finding_img.columns = ["No Finding", "Finding", "Total"]
print(ct_age_finding_img)

Gender x Finding
                No Finding  Finding  Total
Patient Gender                            
F                    12649    10821  23470
M                    16274    13822  30096
All                  28923    24643  53566

Age Category x Finding
              No Finding  Finding  Total
Age Category                            
1-16                1259      899   2158
17-31               5044     3652   8696
32-47               8191     5860  14051
48-62               9684     9122  18806
63-77               4380     4651   9031
78-93                365      459    824
All                28923    24643  53566


In [32]:
### Contingency tables - Patient Dataset

# Gender x Age Category
ct_gender_age = pd.crosstab(binary_patient_dataset["Patient Gender"], binary_patient_dataset["Age Category"], margins=True)
print("Gender x Age Category")
print(ct_gender_age )

# Gender x Finding
ct_gender_finding = pd.crosstab(binary_patient_dataset["Patient Gender"], binary_patient_dataset["Finding"], margins=True)
ct_gender_finding.columns = ["No Finding", "Finding", "Total"]
print( "Gender x Finding")
print(ct_gender_finding)

# Age Category x Finding  
ct_age_finding = pd.crosstab(binary_patient_dataset["Age Category"], binary_patient_dataset["Finding"], margins=True)
ct_age_finding.columns = ["No Finding", "Finding", "Total"]
print("Age Category x Finding")
print(ct_age_finding)

# View Position x Finding (patient-level aggregated view)
ct_view_finding = pd.crosstab(binary_patient_dataset["View Position"], binary_patient_dataset["Finding"], margins=True)
ct_view_finding.columns = ["No Finding", "Finding", "Total"]
print("View Position x Finding")
print(ct_view_finding)


Gender x Age Category
Age Category    1-16  17-31  32-47  48-62  63-77  78-93    All
Patient Gender                                                
F                289   1187   2139   2323    862     86   6886
M                377   1303   2170   2683   1429    152   8114
All              666   2490   4309   5006   2291    238  15000
Gender x Finding
                No Finding  Finding  Total
Patient Gender                            
F                     3766     3120   6886
M                     4286     3828   8114
All                   8052     6948  15000
Age Category x Finding
              No Finding  Finding  Total
Age Category                            
1-16                 414      252    666
17-31               1617      873   2490
32-47               2600     1709   4309
48-62               2399     2607   5006
63-77                947     1344   2291
78-93                 75      163    238
All                 8052     6948  15000
View Position x Finding
               

#### Plots

In [33]:
# Plot 1: Gender x Finding (stacked bar)
gen_find = pd.crosstab(binary_patient_dataset["Patient Gender"], binary_patient_dataset["Finding"], normalize="index") * 100
fig1 = px.bar(gen_find, x=gen_find.index, y=[0, 1], barmode="stack",
             title="Gender × Finding (Patient Dataset)",
             labels={"x": "Gender", "value": "Percentage (%)"},
             color_discrete_map={0: "green", 1: "red"})
fig1.for_each_trace(lambda t: t.update(name="No Finding" if t.name == "0" else "Finding"))
fig1.show()

# Plot 2: Age Category × Finding (stacked bar)
age_find = pd.crosstab(binary_patient_dataset["Age Category"], binary_patient_dataset["Finding"], normalize="index") * 100
fig2 = px.bar(age_find, x=[str(x) for x in age_find.index], y=[0, 1], barmode="stack",
             title="Age Category × Finding (Patient Dataset)",
             labels={"x": "Age Category", "value": "Percentage (%)"},
             color_discrete_map={0: "green", 1: "red"})
fig2.for_each_trace(lambda t: t.update(name="No Finding" if t.name == "0" else "Finding"))
fig2.show()

# Plot 3: View Position × Finding (stacked bar)
view_find = pd.crosstab(binary_patient_dataset["View Position"], binary_patient_dataset["Finding"], normalize="index") * 100
fig3 = px.bar(view_find, x=view_find.index, y=[0, 1], barmode="stack",
             title="View Position × Finding (Patient Dataset)",
             labels={"x": "View Position", "value": "Percentage (%)"},
             color_discrete_map={0: "green", 1: "red"})
fig3.for_each_trace(lambda t: t.update(name="No Finding" if t.name == "0" else "Finding"))
fig3.show()

# Plot 4: Gender × Age Category table
fig4 = go.Figure(data=[go.Table(
    header=dict(values=["Gender"] + list(ct_gender_age.columns),
                fill_color="paleturquoise",
                align="left"),
    cells=dict(values=[ct_gender_age.index.tolist()] + [ct_gender_age[col].tolist() for col in ct_gender_age.columns],
               fill_color="lavender",
               align="left"))])
fig4.update_layout(title="Gender × Age Category (Patient Dataset)", height=400)
fig4.show()


### Correlation Ratio

In [34]:
def correlation_ratio(categorical, numerical):
    categories = categorical.unique()
    overall_mean = numerical.mean()
    overall_var = numerical.var()
    
    between_group_var = 0
    for cat in categories:
        group = numerical[categorical == cat]
        n_group = len(group)
        group_mean = group.mean()
        between_group_var += n_group * (group_mean - overall_mean) ** 2
    
    between_group_var /= len(numerical)
    eta_squared = between_group_var / overall_var if overall_var > 0 else 0
    return eta_squared

### Intersection analysis

In [35]:
binary_patient_dataset["Gender_Age"] = binary_patient_dataset["Patient Gender"] + " - " + binary_patient_dataset["Age Category"].astype(str)

# Compute finding rates 
intersect_rates = binary_patient_dataset.groupby("Gender_Age")["Finding"].agg(["mean", "count"]).reset_index()
intersect_rates.columns = ["Group", "Finding Rate", "Count"]
intersect_rates["Finding Rate"] = (intersect_rates["Finding Rate"] * 100).round(2)
intersect_rates = intersect_rates.sort_values("Finding Rate", ascending=False)


# Bar chart
fig = px.bar(intersect_rates, x="Group", y="Finding Rate", 
             title="Finding Rate by Gender x Age Category (Patient-Level)",
             color="Finding Rate", color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Comparison: Original Dataset Bias vs Patient Dataset

In [36]:
# Code to compute fairness metrics using aif360

from aif360.sklearn.metrics import *
from sklearn.metrics import  balanced_accuracy_score

 
# This method takes lists
def get_metrics(
    y_true, # list or np.array of truth values
    y_pred=None,  # list or np.array of predictions
    prot_attr=None, # list or np.array of protected/sensitive attribute values
    priv_group=1, # value taken by the privileged group
    pos_label=1, # value taken by the positive truth/prediction
    sample_weight=None # list or np.array of weights value,
):
    group_metrics = {}
    group_metrics["base_rate_truth"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["statistical_parity_difference"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["disparate_impact_ratio"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    if not y_pred is None:
        group_metrics["base_rate_preds"] = base_rate(
        y_true=y_pred, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        if len(set(y_pred))>1:
            group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
                y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
            )
        else:
            group_metrics["conditional_demographic_disparity"] =None
        group_metrics["smoothed_edf"] = smoothed_edf(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["balanced_accuracy_score"] = balanced_accuracy_score(
        y_true=y_true, y_pred=y_pred, sample_weight=sample_weight
        )
    return group_metrics

2026-02-18 02:05:57.592969: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-18 02:05:57.976248: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-18 02:06:00.471371: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sipanb/.local/share/mamba/envs/fairness/lib/python3.12/site-packages/keras/src/export

In [37]:
### Compute bias metrics for both datasets

def compute_metrics_for_dataset(df, prot_col, priv_value=1):
    """Compute bias metrics for a given dataset and protected attribute."""
    y = df["Finding"].to_numpy()
    
    # Handle different column names (has_AP vs View_AP)
    if prot_col == "View_AP" and "has_AP" in df.columns:
        A = df["has_AP"].to_numpy()
    else:
        A = df[prot_col].to_numpy()
    
    return get_metrics(y_true=y, prot_attr=A, priv_group=priv_value, pos_label=1)

# Protected attributes to compare
protected_attrs = [
    ("Gender_male", 1),
    ("Age_60plus", 1),
    ("View_AP", 1)  # Will use has_AP for patient-level
]

# Build comparison table
comparison_results = []
for prot_col, priv_val in protected_attrs:
    # Image-level metrics
    metrics_img = compute_metrics_for_dataset(binary_labeled_data, prot_col, priv_val)
    # Patient-level metrics
    metrics_pat = compute_metrics_for_dataset(binary_patient_dataset, prot_col, priv_val)
    
    for metric_name in ["base_rate_truth", "statistical_parity_difference", "disparate_impact_ratio"]:
        comparison_results.append({
            "Protected Attribute": prot_col,
            "Metric": metric_name,
            "Original": metrics_img[metric_name],
            "Patient": metrics_pat[metric_name],
            "Difference": abs(metrics_pat[metric_name] - metrics_img[metric_name])
        })

comparison_df = pd.DataFrame(comparison_results)
print("=== Bias Metrics Comparison: Image-Level vs Patient-Level ===")
print(comparison_df.round(4).to_string(index=False))

=== Bias Metrics Comparison: Image-Level vs Patient-Level ===
Protected Attribute                        Metric  Original  Patient  Difference
        Gender_male               base_rate_truth    0.4600   0.4632      0.0032
        Gender_male statistical_parity_difference    0.0018  -0.0187      0.0205
        Gender_male        disparate_impact_ratio    1.0039   0.9604      0.0435
         Age_60plus               base_rate_truth    0.4600   0.4632      0.0032
         Age_60plus statistical_parity_difference   -0.0797  -0.1672      0.0875
         Age_60plus        disparate_impact_ratio    0.8468   0.7179      0.1289
            View_AP               base_rate_truth    0.4600   0.4632      0.0032
            View_AP statistical_parity_difference   -0.1016  -0.4041      0.3025
            View_AP        disparate_impact_ratio    0.8056   0.4608      0.3448


## Bias Mitigation

In this section we are going to train a simple logistic regression model(not as a final model), to see how the metadata and protected attributes cause bias in predictions. We are going to apply bias mitigation preprocessing methods and see which of them succeed

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from aif360.datasets import StandardDataset
from aif360.algorithms.preprocessing import Reweighing, DisparateImpactRemover
from aif360.algorithms.preprocessing.lfr import LFR
from aif360.metrics import BinaryLabelDatasetMetric

In [40]:
# Prepare features and target (using patient-level data to avoid repeated measurements)
feature_cols = ["Patient Age", "Gender_male", "Age_60plus", "has_AP", "Num_Images"]
X = binary_patient_dataset[feature_cols].copy()
y = binary_patient_dataset["Finding"].copy()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Store protected attributes for fairness evaluation
A_train = X_train[["Gender_male", "Age_60plus", "has_AP"]].copy()
A_test = X_test[["Gender_male", "Age_60plus", "has_AP"]].copy()

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Positive rate (train): {y_train.mean():.4f}")
print(f"Positive rate (test): {y_test.mean():.4f}")

Training set: 10500 samples
Test set: 4500 samples
Positive rate (train): 0.4632
Positive rate (test): 0.4631


In [42]:
# Baseline model (no mitigation)
baseline_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_baseline, target_names=["No Finding", "Finding"]))

Accuracy: 0.7427

Classification Report:
              precision    recall  f1-score   support

  No Finding       0.71      0.88      0.79      2416
     Finding       0.81      0.58      0.68      2084

    accuracy                           0.74      4500
   macro avg       0.76      0.73      0.73      4500
weighted avg       0.76      0.74      0.74      4500



In [43]:
baseline_metrics = {}
for prot_col in ["Gender_male", "Age_60plus", "has_AP"]:
    metrics = get_metrics(
        y_true=y_test.to_numpy(),
        y_pred=y_pred_baseline,
        prot_attr=A_test[prot_col].to_numpy(),
        priv_group=1,
        pos_label=1
    )
    baseline_metrics[prot_col] = metrics
    print(f"\n{prot_col}:")
    for metric_name, value in metrics.items():
        if value is not None:
            print(f"  {metric_name}: {value:.4f}")


Gender_male:
  base_rate_truth: 0.4631
  statistical_parity_difference: -0.0321
  disparate_impact_ratio: 0.9072
  base_rate_preds: 0.3311
  equal_opportunity_difference: -0.0121
  average_odds_difference: -0.0177
  conditional_demographic_disparity: -0.0033
  smoothed_edf: 0.0973
  df_bias_amplification: 0.0321
  balanced_accuracy_score: 0.7315

Age_60plus:
  base_rate_truth: 0.4631
  statistical_parity_difference: -0.2317
  disparate_impact_ratio: 0.5449
  base_rate_preds: 0.3311
  equal_opportunity_difference: -0.1226
  average_odds_difference: -0.1632
  conditional_demographic_disparity: 0.0999
  smoothed_edf: 0.6069
  df_bias_amplification: 0.2725
  balanced_accuracy_score: 0.7315

has_AP:
  base_rate_truth: 0.4631
  statistical_parity_difference: -0.6794
  disparate_impact_ratio: 0.1594
  base_rate_preds: 0.3311
  equal_opportunity_difference: -0.6255
  average_odds_difference: -0.5457
  conditional_demographic_disparity: 0.2594
  smoothed_edf: 1.8354
  df_bias_amplification: 0.

### Reweighing

In [44]:
# Create AIF360 StandardDataset for pre-processing methods
# We'll apply mitigation for Gender_male as primary protected attribute
protected_attr = "Age_60plus"

# Create train/test datasets
train_df = X_train.copy()
train_df["Finding"] = y_train.values
test_df = X_test.copy()
test_df["Finding"] = y_test.values

dataset_train = StandardDataset(
    train_df,
    label_name="Finding",
    favorable_classes=[1],
    protected_attribute_names=[protected_attr],
    privileged_classes=[[1]]
)

dataset_test = StandardDataset(
    test_df,
    label_name="Finding",
    favorable_classes=[1],
    protected_attribute_names=[protected_attr],
    privileged_classes=[[1]]
)

print(f"Train dataset: {dataset_train.features.shape}")
print(f"Test dataset: {dataset_test.features.shape}")

Train dataset: (10500, 5)
Test dataset: (4500, 5)


In [45]:
# Apply Reweighing
privileged_groups = [{protected_attr: 1}]
unprivileged_groups = [{protected_attr: 0}]

RW = Reweighing(unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups)
RW.fit(dataset_train)
dataset_train_rw = RW.transform(dataset_train)

print("=== Reweighing Applied ===")
print(f"Original instance weights sum: {dataset_train.instance_weights.sum():.2f}")
print(f"Reweighted instance weights sum: {dataset_train_rw.instance_weights.sum():.2f}")

# Check dataset metrics before/after
metric_orig = BinaryLabelDatasetMetric(dataset_train, 
                                        unprivileged_groups=unprivileged_groups,
                                        privileged_groups=privileged_groups)
metric_rw = BinaryLabelDatasetMetric(dataset_train_rw,
                                      unprivileged_groups=unprivileged_groups,
                                      privileged_groups=privileged_groups)

print(f"\nStatistical Parity Difference (original): {metric_orig.statistical_parity_difference():.4f}")
print(f"Statistical Parity Difference (reweighted): {metric_rw.statistical_parity_difference():.4f}")
print(f"Disparate Impact (original): {metric_orig.disparate_impact():.4f}")
print(f"Disparate Impact (reweighted): {metric_rw.disparate_impact():.4f}")

=== Reweighing Applied ===
Original instance weights sum: 10500.00
Reweighted instance weights sum: 10500.00

Statistical Parity Difference (original): -0.1688
Statistical Parity Difference (reweighted): 0.0000
Disparate Impact (original): 0.7159
Disparate Impact (reweighted): 1.0000


In [46]:
# Train model with reweighted data
scaler_rw = StandardScaler()
X_train_rw_scaled = scaler_rw.fit_transform(dataset_train_rw.features)
X_test_scaled = scaler_rw.transform(dataset_test.features)

rw_model = LogisticRegression(max_iter=1000, random_state=42)
rw_model.fit(X_train_rw_scaled, dataset_train_rw.labels.ravel(), 
             sample_weight=dataset_train_rw.instance_weights)

y_pred_rw = rw_model.predict(X_test_scaled)

print("=== Reweighting Model Performance ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rw):.4f}")
print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")

# Fairness metrics
rw_metrics = get_metrics(
    y_true=y_test.to_numpy(),
    y_pred=y_pred_rw,
    prot_attr=A_test[protected_attr].to_numpy(),
    priv_group=1,
    pos_label=1
)
print(f"\n{protected_attr} Fairness Metrics (Reweighting):")
for metric_name, value in rw_metrics.items():
    if value is not None:
        print(f"  {metric_name}: {value:.4f}")

=== Reweighting Model Performance ===
Accuracy: 0.7376
Baseline Accuracy: 0.7427

Age_60plus Fairness Metrics (Reweighting):
  base_rate_truth: 0.4631
  statistical_parity_difference: -0.0564
  disparate_impact_ratio: 0.8405
  base_rate_preds: 0.3104
  equal_opportunity_difference: 0.0550
  average_odds_difference: 0.0131
  conditional_demographic_disparity: 0.0252
  smoothed_edf: 0.1739
  df_bias_amplification: -0.1605
  balanced_accuracy_score: 0.7248


### Disparate Impact Remover

In [47]:
# Apply Disparate Impact Remover
# Note: DIR only supports fit_transform(), not separate transform()
DIR = DisparateImpactRemover(repair_level=1.0, sensitive_attribute=protected_attr)
dataset_train_dir = DIR.fit_transform(dataset_train)
dataset_test_dir = DIR.fit_transform(dataset_test)

print("=== Disparate Impact Remover Applied ===")
print(f"Original features shape: {dataset_train.features.shape}")
print(f"Transformed features shape: {dataset_train_dir.features.shape}")

# Train model on transformed data
scaler_dir = StandardScaler()
X_train_dir_scaled = scaler_dir.fit_transform(dataset_train_dir.features)
X_test_dir_scaled = scaler_dir.transform(dataset_test_dir.features)

dir_model = LogisticRegression(max_iter=1000, random_state=42)
dir_model.fit(X_train_dir_scaled, dataset_train_dir.labels.ravel())

y_pred_dir = dir_model.predict(X_test_dir_scaled)

print(f"\n=== DIR Model Performance ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dir):.4f}")
print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")

# Fairness metrics
dir_metrics = get_metrics(
    y_true=y_test.to_numpy(),
    y_pred=y_pred_dir,
    prot_attr=A_test[protected_attr].to_numpy(),
    priv_group=1,
    pos_label=1
)
print(f"\n{protected_attr} Fairness Metrics (DIR):")
for metric_name, value in dir_metrics.items():
    if value is not None:
        print(f"  {metric_name}: {value:.4f}")

=== Disparate Impact Remover Applied ===
Original features shape: (10500, 5)
Transformed features shape: (10500, 5)

=== DIR Model Performance ===
Accuracy: 0.7369
Baseline Accuracy: 0.7427

Age_60plus Fairness Metrics (DIR):
  base_rate_truth: 0.4631
  statistical_parity_difference: -0.2280
  disparate_impact_ratio: 0.5497
  base_rate_preds: 0.3311
  equal_opportunity_difference: -0.1222
  average_odds_difference: -0.1612
  conditional_demographic_disparity: 0.0983
  smoothed_edf: 0.5981
  df_bias_amplification: 0.2638
  balanced_accuracy_score: 0.7257


### LFR

In [48]:
# Apply LFR (Learning Fair Representations)
# Note: LFR can be slow, using limited iterations for demo
LFR_model = LFR(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups,
    k=5,  # Number of prototypes
    Ax=0.01,  # Reconstruction loss weight
    Ay=1.0,   # Prediction loss weight  
    Az=50.0,  # Fairness loss weight
    verbose=1
)

print("Training LFR (this may take a few minutes)...")
LFR_model = LFR_model.fit(dataset_train, maxiter=2000, maxfun=2000)

# Transform datasets
dataset_train_lfr = LFR_model.transform(dataset_train)
dataset_test_lfr = LFR_model.transform(dataset_test)

print(f"\n=== LFR Applied ===")
print(f"Original features shape: {dataset_train.features.shape}")
print(f"LFR features shape: {dataset_train_lfr.features.shape}")

Training LFR (this may take a few minutes)...
step: 0, loss: 13.389941541382536, L_x: 1261.0335284516568,  L_y: 0.7139164138070824,  L_z: 0.0013137968611777017
step: 250, loss: 8.367304215029225, L_x: 366.26583616379867,  L_y: 0.7065598680825118,  L_z: 0.07996171970617455
step: 500, loss: 6.1027371456869135, L_x: 329.1168365427235,  L_y: 0.6881789961863459,  L_z: 0.04246779568146665
step: 750, loss: 3.9366158928027666, L_x: 168.68956994456806,  L_y: 0.7356853384420982,  L_z: 0.030280697098299753
step: 1000, loss: 2.222458966925745, L_x: 152.19119779614232,  L_y: 0.6919580049240905,  L_z: 0.0001717796808046232
step: 1250, loss: 2.061504860849802, L_x: 136.93769230184932,  L_y: 0.6904494361848208,  L_z: 3.3570032929749157e-05
step: 1500, loss: 2.061501280203583, L_x: 136.96604483043265,  L_y: 0.6904481984503721,  L_z: 2.78526689776864e-05
step: 1750, loss: 2.061501079160552, L_x: 136.96900716224238,  L_y: 0.6904478358307327,  L_z: 2.7263434147917696e-05
step: 2000, loss: 2.06150000334010

In [49]:
# Train model on LFR-transformed data
# Note: LFR.transform() overwrites labels with its own predictions,
# so we use the original labels for training instead.
scaler_lfr = StandardScaler()
X_train_lfr_scaled = scaler_lfr.fit_transform(dataset_train_lfr.features)
X_test_lfr_scaled = scaler_lfr.transform(dataset_test_lfr.features)

lfr_clf = LogisticRegression(max_iter=1000, random_state=42)
lfr_clf.fit(X_train_lfr_scaled, y_train.values)  # Use original labels, not LFR-transformed ones

y_pred_lfr = lfr_clf.predict(X_test_lfr_scaled)

print("=== LFR Model Performance ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lfr):.4f}")
print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")

# Fairness metrics
lfr_metrics = get_metrics(
    y_true=y_test.to_numpy(),
    y_pred=y_pred_lfr,
    prot_attr=A_test[protected_attr].to_numpy(),
    priv_group=1,
    pos_label=1
)
print(f"\n{protected_attr} Fairness Metrics (LFR):")
for metric_name, value in lfr_metrics.items():
    if value is not None:
        print(f"  {metric_name}: {value:.4f}")

=== LFR Model Performance ===
Accuracy: 0.5369
Baseline Accuracy: 0.7427

Age_60plus Fairness Metrics (LFR):
  base_rate_truth: 0.4631
  statistical_parity_difference: 0.0000
  disparate_impact_ratio: 0.0000
  base_rate_preds: 0.0000
  equal_opportunity_difference: 0.0000
  average_odds_difference: 0.0000
  smoothed_edf: 1.1976
  df_bias_amplification: 0.8633
  balanced_accuracy_score: 0.5000


/home/sipanb/.local/share/mamba/envs/fairness/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Ratio is ill-defined and being set to 0.0 due to no predicted privileged samples. Use `zero_division` parameter to control this behavior.



### Comparaison

In [50]:
# Create comparison dataframe
comparison_data = []
methods = [
    ("Baseline", baseline_metrics[protected_attr], accuracy_score(y_test, y_pred_baseline)),
    ("Reweighing", rw_metrics, accuracy_score(y_test, y_pred_rw)),
    ("Disparate Impact Remover", dir_metrics, accuracy_score(y_test, y_pred_dir)),
    ("LFR", lfr_metrics, accuracy_score(y_test, y_pred_lfr))
]

for method_name, metrics, accuracy in methods:
    comparison_data.append({
        "Method": method_name,
        "Accuracy": accuracy,
        "Statistical Parity Diff": metrics.get("statistical_parity_difference"),
        "Disparate Impact Ratio": metrics.get("disparate_impact_ratio"),
        "Equal Opportunity Diff": metrics.get("equal_opportunity_difference"),
        "Average Odds Diff": metrics.get("average_odds_difference")
    })

comparison_df_methods = pd.DataFrame(comparison_data)
print(f"=== Method Comparison (Protected Attribute: {protected_attr}) ===")
print(comparison_df_methods.round(4).to_string(index=False))

=== Method Comparison (Protected Attribute: Age_60plus) ===
                  Method  Accuracy  Statistical Parity Diff  Disparate Impact Ratio  Equal Opportunity Diff  Average Odds Diff
                Baseline    0.7427                  -0.2317                  0.5449                 -0.1226            -0.1632
              Reweighing    0.7376                  -0.0564                  0.8405                  0.0550             0.0131
Disparate Impact Remover    0.7369                  -0.2280                  0.5497                 -0.1222            -0.1612
                     LFR    0.5369                   0.0000                  0.0000                  0.0000             0.0000


In [51]:
# Visualize comparison
fig = make_subplots(rows=1, cols=2, subplot_titles=["Accuracy", "|Statistical Parity Difference|"])

methods_list = comparison_df_methods["Method"].tolist()

# Accuracy
fig.add_trace(go.Bar(x=methods_list, y=comparison_df_methods["Accuracy"], 
                     name="Accuracy", marker_color="blue"), row=1, col=1)

# Statistical Parity Difference (closer to 0 is better)
fig.add_trace(go.Bar(x=methods_list, y=comparison_df_methods["Statistical Parity Diff"].abs(),
                     name="|SPD|", marker_color="red"), row=1, col=2)

fig.update_layout(title=f"Pre-Processing Methods Comparison (Protected: {protected_attr})", 
                  height=400, showlegend=False)
fig.add_hline(y=0, line_dash="dash", line_color="green", row=1, col=2)
fig.show()

## Conclusion

In this notebook we explored the Chest X-Ray dataset. We found indicators of existance of bias. We did our anlysis in two fonts: \
- Keeping original dataset format, where each row corresponds to an image
- Patient centric dataset, where each row corresponds to a patient


In our analysis we found indicators that there is bias present in the dataset, identified that it is more expressed for the protected attribute that has information about age > 60. 
In the final section we applied bias mitigation methods to the patient centric dataset, as it has more expressed bias, and we compared the different bias mitigation methods. We found that Reweighing is a good choice to reduce bias by keeping the same level of accuracy, and LFR allows for a more reduction, but looses a lot of accuracy. 

In our case, the model is just trained on metadata, so expecting a lot of accuracy from it is a wise choice. On the contrary, accuracy of the model trained on just the metadata can show the bias exisiting in the dataset, so in case of LFR, as the accuracy is near 50 percent, we can deduce that it will be a good way of reducing the bias for the final model trained on actual data. 